# Collecting Data on AI Skills Repositories from GitHub

*Course:* MGS 3001 WHS01 — Python Programming for Business  
*Session:* Week 9-1, API-Based Data Collection  
*Date:* 21 April 2026  
*Instructor:* Dawei Chen

---

### What This Notebook Does

This notebook uses the **GitHub REST API** to collect structured data on repositories related to **AI skills and agent skills**.

We search across multiple keywords, extract research-relevant fields, handle pagination, deduplicate results, and export to CSV.

### The Pipeline

```
Define Scope → Craft Prompt → API Call → JSON Response → Parse Fields → Paginate → Deduplicate → CSV
```

### Before You Start

You need a **GitHub Personal Access Token**. If you haven't set one up yet:

1. Go to github.com → Settings → Developer settings → Personal access tokens → Tokens (classic)
2. Generate a new token with `public_repo` scope
3. Copy the token and paste it in Cell 2 below

⚠️ **Never share your token or commit it to GitHub.**

---
## Cell 1: Import Libraries

We need:
- `requests` — to send HTTP requests to the GitHub API
- `pandas` — to organize data into a table (DataFrame) and export to CSV
- `time` — to add delays between requests (be a polite scraper!)
- `json` — to inspect raw JSON responses

In [1]:
import requests
import pandas as pd
import time
import json

print("Libraries imported successfully!")

Libraries imported successfully!


---
## Cell 2: Set Up Authentication

Paste your GitHub Personal Access Token below. The token is sent in the HTTP **headers** with every request, so GitHub knows who you are and gives you a higher rate limit (5,000 requests/hour instead of 60).

⚠️ **Replace the placeholder with your actual token.**

In [ ]:
# ⚠️ Paste your GitHub Personal Access Token here
# IMPORTANT: Never share this token or upload it to a public repository
GITHUB_TOKEN = "your_token_here"  # <-- Replace this!

# Set up HTTP headers for authentication
# The "Authorization" header tells GitHub who we are
# The "Accept" header tells GitHub we want JSON data (API version 3)
headers = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json"
}

# Quick check: verify the token works
test = requests.get("https://api.github.com/rate_limit", headers=headers)
if test.status_code == 200:
    rate_info = test.json()
    remaining = rate_info['resources']['search']['remaining']
    limit = rate_info['resources']['search']['limit']
    print(f"✅ Token is valid! Search API: {remaining}/{limit} requests remaining.")
else:
    print(f"❌ Token check failed. Status code: {test.status_code}")
    print("   Please check your token and try again.")

✅ Token is valid! Search API: 30/30 requests remaining.


---
## Cell 3: Define the Search Scope

This is the **Delegation** step from the 4D framework — we define the problem before asking AI or writing code.

**Key decisions:**
- Which keywords to search? (We use multiple to avoid missing relevant repos)
- How many results per keyword? (GitHub API returns max 1,000 per query)
- How to sort? (By stars = most popular first)

**Why multiple keywords?**
- "AI skills" vs "AI skill" (plural vs singular) → different results
- "agent skills" vs "agent-skills" (space vs dash) → different results
- GitHub search is case-insensitive, but space/dash matters for topic tags

In [3]:
# --- Search Configuration ---

# Multiple keywords to cast a wider net
# Each keyword will be searched separately, then results are combined and deduplicated
KEYWORDS = [
    "AI skills",
    "AI skill",
    "agent skills",
    "agent-skills",
    "claude skills",
    "LLM skills",
]

# API endpoint for searching repositories
API_URL = "https://api.github.com/search/repositories"

# Maximum results to collect per keyword
# GitHub Search API returns max 1,000 results per query
MAX_RESULTS_PER_KEYWORD = 200

# Results per page (max 100 for GitHub)
PER_PAGE = 100

# Sort by stars (most popular first)
SORT_BY = "stars"

# Calculate how many pages we need per keyword
max_pages = (MAX_RESULTS_PER_KEYWORD + PER_PAGE - 1) // PER_PAGE

print(f"Keywords to search: {KEYWORDS}")
print(f"Max results per keyword: {MAX_RESULTS_PER_KEYWORD}")
print(f"Results per page: {PER_PAGE}")
print(f"Pages per keyword: {max_pages}")
print(f"Sort by: {SORT_BY}")

Keywords to search: ['AI skills', 'AI skill', 'agent skills', 'agent-skills', 'claude skills', 'LLM skills']
Max results per keyword: 200
Results per page: 100
Pages per keyword: 2
Sort by: stars


---
## Cell 4: Make the First API Call (Single Page)

Before running the full collection loop, let's make **one API call** to understand what the response looks like.

This is the **Discernment** step — we inspect the raw output before trusting it.

In [4]:
# Make a single test request with the first keyword
test_params = {
    "q": KEYWORDS[0],       # Search query
    "sort": SORT_BY,         # Sort by stars
    "order": "desc",         # Descending (most stars first)
    "per_page": 5,           # Just 5 results for inspection
    "page": 1                # First page
}

response = requests.get(API_URL, headers=headers, params=test_params)

print(f"Status Code: {response.status_code}")
print(f"URL requested: {response.url}")
print()

if response.status_code == 200:
    data = response.json()
    print(f"Total results found for '{KEYWORDS[0]}': {data['total_count']}")
    print(f"Results returned in this page: {len(data['items'])}")
else:
    print(f"Error: {response.status_code}")
    print(response.json().get('message', 'Unknown error'))

Status Code: 200
URL requested: https://api.github.com/search/repositories?q=AI+skills&sort=stars&order=desc&per_page=5&page=1

Total results found for 'AI skills': 46496
Results returned in this page: 5


---
## Cell 5: Inspect the Raw JSON Response

Let's look at what the API actually returns. Notice:
- The response has a top-level `total_count` and an `items` array
- Each item has many fields — some are simple values (strings, numbers)
- Some fields are **nested objects** (e.g., `owner` is an object with `login` and `type`)
- Some fields are **arrays** (e.g., `topics` is a list of strings)

We'll learn to handle nested and array data properly in Thursday's session.

In [5]:
# Look at the first repository in detail
if response.status_code == 200 and len(data['items']) > 0:
    first_repo = data['items'][0]
    
    # Print the full JSON for the first repo (pretty-printed)
    print("=== Raw JSON for the first repository ===")
    print(json.dumps(first_repo, indent=2, ensure_ascii=False)[:2000])  # Truncate for readability
    print("\n... (truncated)")
else:
    print("No data to inspect. Check the API call above.")

=== Raw JSON for the first repository ===
{
  "id": 132464395,
  "node_id": "MDEwOlJlcG9zaXRvcnkxMzI0NjQzOTU=",
  "name": "JavaGuide",
  "full_name": "Snailclimb/JavaGuide",
  "private": false,
  "owner": {
    "login": "Snailclimb",
    "id": 29880145,
    "node_id": "MDQ6VXNlcjI5ODgwMTQ1",
    "avatar_url": "https://avatars.githubusercontent.com/u/29880145?v=4",
    "gravatar_id": "",
    "url": "https://api.github.com/users/Snailclimb",
    "html_url": "https://github.com/Snailclimb",
    "followers_url": "https://api.github.com/users/Snailclimb/followers",
    "following_url": "https://api.github.com/users/Snailclimb/following{/other_user}",
    "gists_url": "https://api.github.com/users/Snailclimb/gists{/gist_id}",
    "starred_url": "https://api.github.com/users/Snailclimb/starred{/owner}{/repo}",
    "subscriptions_url": "https://api.github.com/users/Snailclimb/subscriptions",
    "organizations_url": "https://api.github.com/users/Snailclimb/orgs",
    "repos_url": "https://api.

In [ ]:
# Let's look at just the fields we care about for the first repo
if response.status_code == 200 and len(data['items']) > 0:
    repo = data['items'][0]
    
    print("=== Fields we want to extract ===")
    print(f"  full_name:         {repo['full_name']}")
    print(f"  description:       {repo.get('description', 'N/A')[:80]}...")  # Truncate long descriptions
    print(f"  stargazers_count:  {repo['stargazers_count']}")
    print(f"  forks_count:       {repo['forks_count']}")
    print(f"  open_issues_count: {repo['open_issues_count']}")
    print(f"  created_at:        {repo['created_at']}")
    print(f"  updated_at:        {repo['updated_at']}")
    print(f"  language:          {repo.get('language', 'N/A')}")
    print()
    
    # Nested fields — need special handling
    print("=== Nested fields ===")
    print(f"  owner.login:       {repo['owner']['login']}")
    print(f"  owner.type:        {repo['owner']['type']}")
    
    # License can be None (null) for some repos!
    license_name = repo['license']['name'] if repo.get('license') else 'No license'
    print(f"  license.name:      {license_name}")
    print()
    
    # Array field
    print("=== Array field ===")
    print(f"  topics:            {repo.get('topics', [])}")

---
## Cell 6: Define the Field Extraction Function

Now we write a function that extracts the fields we want from each repository's JSON.

**Why a function?** Because we'll call it for every repository across all keywords and pages. Writing it once and reusing it keeps the code clean.

**Important:** Some fields can be `None` (null) — for example, a repo might not have a license or a description. We use `.get()` with default values to handle this gracefully.

In [6]:
def extract_repo_info(repo):
    """
    Extract research-relevant fields from a single repository's JSON.
    
    Each field maps to a research construct:
    - full_name          → identity
    - description        → what the skill does (text analysis later)
    - stargazers_count   → market attention / demand proxy
    - forks_count        → adoption / reuse signal
    - open_issues_count  → user engagement / quality signal
    - created_at         → entry timing
    - updated_at         → maintenance activity
    - language           → primary programming language
    - license_name       → commercial openness
    - owner_login        → who built it
    - owner_type         → individual vs. organization
    - topics             → self-declared categories
    """
    return {
        "full_name":         repo.get("full_name", ""),
        "description":       repo.get("description", ""),
        "stargazers_count":  repo.get("stargazers_count", 0),
        "forks_count":       repo.get("forks_count", 0),
        "open_issues_count": repo.get("open_issues_count", 0),
        "created_at":        repo.get("created_at", ""),
        "updated_at":        repo.get("updated_at", ""),
        "language":          repo.get("language", ""),
        "license_name":      repo["license"]["name"] if repo.get("license") else "",
        "owner_login":       repo["owner"]["login"] if repo.get("owner") else "",
        "owner_type":        repo["owner"]["type"] if repo.get("owner") else "",
        "topics":            ", ".join(repo.get("topics", [])),  # Convert list to comma-separated string
        "html_url":          repo.get("html_url", ""),           # Link to the repo (useful for reference)
    }

# Test the function on the first repo
if response.status_code == 200 and len(data['items']) > 0:
    test_extracted = extract_repo_info(data['items'][0])
    print("=== Extracted fields ===")
    for key, value in test_extracted.items():
        display_value = str(value)[:80] + "..." if len(str(value)) > 80 else str(value)
        print(f"  {key:22s} {display_value}")

=== Extracted fields ===
  full_name              Snailclimb/JavaGuide
  description            Java 面试 & 后端通用面试指南，覆盖计算机基础、数据库、分布式、高并发、系统设计与 AI 应用开发
  stargazers_count       155151
  forks_count            46149
  open_issues_count      64
  created_at             2018-05-07T13:27:00Z
  updated_at             2026-04-23T02:04:49Z
  language               Java
  license_name           Apache License 2.0
  owner_login            Snailclimb
  owner_type             User
  topics                 agent, context-engineering, interview, java, jvm, mcp, mysql, redis, redisson, s...
  html_url               https://github.com/Snailclimb/JavaGuide


---
## Cell 7: Collect Data Across All Keywords (with Pagination)

Now we run the full collection:
1. Loop through each keyword
2. For each keyword, loop through pages (pagination)
3. Extract fields from each repository
4. Add a `search_keyword` column so we know which keyword found each repo
5. Add a delay between requests to be polite (rate limiting)

This is the **automated version** of what a human would do manually: search → open each repo → copy data → repeat.

In [7]:
# Collect all results
all_repos = []

for keyword in KEYWORDS:
    print(f"\n🔍 Searching for: '{keyword}'")
    keyword_count = 0
    
    for page in range(1, max_pages + 1):
        # Set up the request parameters
        params = {
            "q": keyword,
            "sort": SORT_BY,
            "order": "desc",
            "per_page": PER_PAGE,
            "page": page
        }
        
        # Make the API request
        response = requests.get(API_URL, headers=headers, params=params)
        
        # Check for errors
        if response.status_code != 200:
            print(f"  ⚠️  Error on page {page}: status {response.status_code}")
            error_msg = response.json().get('message', 'Unknown error')
            print(f"      Message: {error_msg}")
            
            # If rate limited, wait and retry
            if response.status_code == 403 or response.status_code == 429:
                print("      Rate limited. Waiting 60 seconds...")
                time.sleep(60)
                continue
            else:
                break
        
        # Parse the response
        data = response.json()
        items = data.get("items", [])
        
        # If no more results, stop pagination for this keyword
        if len(items) == 0:
            print(f"  No more results on page {page}. Moving to next keyword.")
            break
        
        # Extract fields from each repository
        for repo in items:
            repo_info = extract_repo_info(repo)
            repo_info["search_keyword"] = keyword  # Track which keyword found this repo
            all_repos.append(repo_info)
            keyword_count += 1
        
        print(f"  Page {page}: collected {len(items)} repos (total for this keyword: {keyword_count})")
        
        # Be polite: wait between requests to avoid hitting rate limits
        time.sleep(2)
    
    print(f"  ✅ Finished '{keyword}': {keyword_count} repos collected")

print(f"\n{'='*50}")
print(f"Total repos collected (before deduplication): {len(all_repos)}")


🔍 Searching for: 'AI skills'
  Page 1: collected 100 repos (total for this keyword: 100)
  Page 2: collected 100 repos (total for this keyword: 200)
  ✅ Finished 'AI skills': 200 repos collected

🔍 Searching for: 'AI skill'
  Page 1: collected 100 repos (total for this keyword: 100)
  Page 2: collected 100 repos (total for this keyword: 200)
  ✅ Finished 'AI skill': 200 repos collected

🔍 Searching for: 'agent skills'
  Page 1: collected 100 repos (total for this keyword: 100)
  Page 2: collected 100 repos (total for this keyword: 200)
  ✅ Finished 'agent skills': 200 repos collected

🔍 Searching for: 'agent-skills'
  Page 1: collected 100 repos (total for this keyword: 100)
  Page 2: collected 100 repos (total for this keyword: 200)
  ✅ Finished 'agent-skills': 200 repos collected

🔍 Searching for: 'claude skills'
  Page 1: collected 100 repos (total for this keyword: 100)
  Page 2: collected 100 repos (total for this keyword: 200)
  ✅ Finished 'claude skills': 200 repos collected

🔍

---
## Cell 8: Deduplicate Results

Since we searched multiple keywords, the **same repository** may appear in results for different keywords. For example, a repo about "AI agent skills" might match both "AI skills" and "agent skills".

We deduplicate by `full_name` (which is unique for each repository on GitHub), keeping the first occurrence.

In [8]:
# Convert to DataFrame
df = pd.DataFrame(all_repos)

print(f"Before deduplication: {len(df)} rows")

# Count how many repos were found by multiple keywords
duplicate_count = df.duplicated(subset=['full_name'], keep='first').sum()
print(f"Duplicate repos (found by multiple keywords): {duplicate_count}")

# Remove duplicates, keeping the first occurrence
df = df.drop_duplicates(subset=['full_name'], keep='first')

print(f"After deduplication: {len(df)} unique repos")

# Reset the index
df = df.reset_index(drop=True)

print(f"\nDataset shape: {df.shape[0]} rows × {df.shape[1]} columns")

Before deduplication: 1200 rows
Duplicate repos (found by multiple keywords): 643
After deduplication: 557 unique repos

Dataset shape: 557 rows × 14 columns


---
## Cell 9: Preview the Dataset

Let's look at what we collected. This is another **Discernment** checkpoint — does the data look right? Are there any obvious problems?

In [9]:
# Show the first 10 rows
print("=== First 10 repositories ===")
df.head(10)

=== First 10 repositories ===


,full_name,description,stargazers_count,forks_count,open_issues_count,created_at,updated_at,language,license_name,owner_login,owner_type,topics,html_url,search_keyword
0,Snailclimb/JavaGuide,Java 面试 & 后端通用面试指南，覆盖计算机基础、数据库、分布式、高并发、系统设计与 A...,155152,46149,64,2018-05-07T13:27:00Z,2026-04-23T02:59:24Z,Java,Apache License 2.0,Snailclimb,User,"agent, context-engineering, interview, java, j...",https://github.com/Snailclimb/JavaGuide,AI skills
1,nextlevelbuilder/ui-ux-pro-max-skill,An AI SKILL that provide design intelligence f...,69317,7092,118,2025-11-30T11:36:31Z,2026-04-23T03:04:18Z,Python,MIT License,nextlevelbuilder,Organization,"ai-skills, antigravity, claude, claude-code, c...",https://github.com/nextlevelbuilder/ui-ux-pro-...,AI skills
2,bytedance/deer-flow,An open-source long-horizon SuperAgent harness...,63457,8246,722,2025-05-07T02:50:19Z,2026-04-23T03:02:12Z,Python,MIT License,bytedance,Organization,"agent, agentic, agentic-framework, agentic-wor...",https://github.com/bytedance/deer-flow,AI skills
3,ComposioHQ/awesome-claude-skills,"A curated list of awesome Claude Skills, resou...",55694,5978,546,2025-10-17T07:15:01Z,2026-04-23T03:06:04Z,Python,,ComposioHQ,Organization,"agent-skills, ai-agents, antigravity, automati...",https://github.com/ComposioHQ/awesome-claude-s...,AI skills
4,jeecgboot/JeecgBoot,"一款 AI 驱动的低代码平台，提供""零代码""与""代码生成""双模式——零代码模式一句话搭建系统...",45931,15935,13,2018-11-26T10:40:00Z,2026-04-23T02:51:25Z,Java,Apache License 2.0,jeecgboot,Organization,"activiti, agent, ai, aiflow, antd, codegenerat...",https://github.com/jeecgboot/JeecgBoot,AI skills
5,CherryHQ/cherry-studio,"AI productivity studio with smart chat, autono...",44101,4166,1020,2024-05-24T01:56:26Z,2026-04-23T03:04:53Z,TypeScript,GNU Affero General Public License v3.0,CherryHQ,Organization,"agency-agents, ai-agent, claude-code, codex, o...",https://github.com/CherryHQ/cherry-studio,AI skills
6,JuliusBrussee/caveman,🪨 why use many token when few token do trick —...,43691,2273,142,2026-04-04T10:03:00Z,2026-04-23T03:05:26Z,Python,MIT License,JuliusBrussee,User,"ai, anthropic, caveman, claude, claude-code, l...",https://github.com/JuliusBrussee/caveman,AI skills
7,zhayujie/CowAgent,CowAgent (chatgpt-on-wechat) 是基于大模型的超级AI助理，能主动...,43640,9976,353,2022-08-07T08:33:41Z,2026-04-23T02:26:04Z,Python,MIT License,zhayujie,User,"ai, ai-agent, chatgpt-on-wechat, claude, deeps...",https://github.com/zhayujie/CowAgent,AI skills
8,santifer/career-ops,AI-powered job search system built on Claude C...,38557,7831,115,2026-04-04T18:21:18Z,2026-04-23T03:06:20Z,JavaScript,MIT License,santifer,User,"ai-agent, anthropic, automation, career, claud...",https://github.com/santifer/career-ops,AI skills
9,safishamsi/graphify,"AI coding assistant skill (Claude Code, Codex,...",33174,3672,159,2026-04-03T15:49:07Z,2026-04-23T03:06:12Z,Python,MIT License,safishamsi,User,"antigravity, claude-code, codex, gemini, graph...",https://github.com/safishamsi/graphify,AI skills


In [ ]:
# Quick summary statistics
print("=== Column names and data types ===")
print(df.dtypes)
print()

print("=== Basic statistics for numeric columns ===")
df[['stargazers_count', 'forks_count', 'open_issues_count']].describe()

In [ ]:
# Check for missing values
print("=== Missing values per column ===")
# Count empty strings and None as missing
missing = df.replace('', pd.NA).isna().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values!")
print()

# Which keywords found the most repos?
print("=== Repos found per search keyword ===")
print(df['search_keyword'].value_counts())
print()

# Owner type distribution
print("=== Owner type distribution ===")
print(df['owner_type'].value_counts())

---
## Cell 10: Export to CSV

Save the dataset to a CSV file. This file will be used in future sessions for:
- JSON & semi-structured data processing (Thursday 4/23)
- Data cleaning (5/07)
- Exploratory data analysis (5/19)
- Visualization (5/21)
- Regression analysis (5/26–5/28)

In [ ]:
# Export to CSV
output_filename = "github_ai_skills_repos.csv"
df.to_csv(output_filename, index=False, encoding='utf-8-sig')  # utf-8-sig for Excel compatibility

print(f"✅ Dataset saved to: {output_filename}")
print(f"   Rows: {len(df)}")
print(f"   Columns: {len(df.columns)}")
print(f"   Columns: {list(df.columns)}")

---
## 🎯 Hands-on Tasks

Now it's your turn! Try the following modifications:

### Task 1: Run the notebook as-is
Run all cells above and inspect the output CSV file.

### Task 2: Change the keywords
Go back to **Cell 3** and modify the `KEYWORDS` list. For example:
```python
KEYWORDS = [
    "workflow automation",
    "MCP skills",
    "AI agent tools",
    # Add keywords related to your own research topic!
]
```
Then re-run Cells 3 through 10.

### Task 3: Add or remove fields
Go back to **Cell 6** and modify the `extract_repo_info()` function. For example:
```python
# Add these fields:
"size":              repo.get("size", 0),           # Repo size in KB
"has_wiki":          repo.get("has_wiki", False),   # Does it have a wiki?
"watchers_count":    repo.get("watchers_count", 0), # Number of watchers
```

### Task 4 (stretch): Add date filters
In **Cell 3**, modify the keyword to include a date filter:
```python
KEYWORDS = [
    "AI skills created:>2024-01-01",   # Only repos created after Jan 2024
]
```
This uses GitHub's search qualifiers. See: https://docs.github.com/en/search-github/searching-on-github/searching-for-repositories

---
## 📝 Reflection (4D Framework)

After completing the tasks, think about:

- **Delegation:** What parts of this process could only a human decide? What parts did the code handle?
- **Description:** How did the specificity of our keywords and field definitions affect the quality of the data?
- **Discernment:** Did the collected data match what you expected? Any surprises or problems?
- **Diligence:** Did we respect rate limits? Is our search scope reasonable and defensible for research?

In [ ]:
# Optional: record your reflections
reflection = {
    "delegation":  "",
    "description": "",
    "discernment": "",
    "diligence":   ""
}

reflection